# RAG System - End-to-End Demo

This notebook demonstrates the complete RAG pipeline:
1. **PDF Ingestion & Parsing** - Extract text, clean, and chunk with metadata
2. **Embedding Generation** - Convert chunks to vectors (TBD)
3. **Vector Storage** - Store in FAISS index (TBD)
4. **Query & Retrieval** - Semantic search and re-ranking (TBD)
5. **LLM Response** - Generate answers with citations (TBD)
6. **Evaluation** - Metrics on test questions (TBD)

---
## Setup

In [1]:
# Imports
import numpy as np
from pathlib import Path
import json

from rag_system.models import DocumentChunk, ParsedDocument
from rag_system.parser import DocumentParser, ParserConfig
from rag_system.embeddings import Embedder, EmbeddingConfig

In [2]:
# Project paths
PROJECT_ROOT = Path.cwd()
PDFS_DIR = PROJECT_ROOT / "pdfs"

print(f"Project root: {PROJECT_ROOT}")
print(f"PDFs directory: {PDFS_DIR}")

Project root: c:\Development\git\rag_achiles
PDFs directory: c:\Development\git\rag_achiles\pdfs


---
## 1. PDF Ingestion & Parsing

### Available Documents

We have 9 Madrid city council meeting summaries in Spanish.

In [3]:
# List all PDFs
pdf_files = sorted(PDFS_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files:\n")
for i, pdf_path in enumerate(pdf_files, 1):
    size_mb = pdf_path.stat().st_size / (1024 * 1024)
    print(f"{i}. {pdf_path.name:<50} ({size_mb:.2f} MB)")

total_size_mb = sum(p.stat().st_size for p in pdf_files) / (1024 * 1024)
print(f"\nTotal size: {total_size_mb:.2f} MB")

Found 9 PDF files:

1. Resumen_8_reunion_grupo_motor_noviembre2025.pdf    (0.45 MB)
2. Resumen_9_reunion_marzo2026.pdf                    (0.23 MB)
3. Resumen_grupo-motor130924.pdf                      (0.51 MB)
4. resumen_grupo-motor240520.pdf                      (0.47 MB)
5. resumen_grupo_motor110424.pdf                      (1.31 MB)
6. Resumen_grupo_motor_041024.pdf                     (0.70 MB)
7. Resumen_grupo_motor_14032025.pdf                   (0.37 MB)
8. resumengrupomotor260224.pdf                        (0.65 MB)
9. ResumenReunionGA_20231124.pdf                      (1.11 MB)

Total size: 5.81 MB


### Parser Configuration

Using **RecursiveCharacterTextSplitter** with:
- **Chunk size**: 512 characters (≈ tokens)
- **Overlap**: 50 characters
- **Separators**: `["\n\n", "\n", " ", ""]` (paragraphs → lines → words → chars)

This approach respects semantic boundaries (paragraphs, sentences) instead of cutting mid-sentence.

In [4]:
# Create parser with default configuration
config = ParserConfig(
    chunk_size=512,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

parser = DocumentParser(config=config)

print("Parser Configuration:")
print(json.dumps(config.model_dump(), indent=2))

Parser Configuration:
{
  "chunk_size": 512,
  "chunk_overlap": 50,
  "separators": [
    "\n\n",
    "\n",
    " ",
    ""
  ]
}


### Parse Single PDF (Example)

Let's parse one document to see the process in detail.

In [5]:
# Parse first PDF
sample_pdf = pdf_files[0]
print(f"Parsing: {sample_pdf.name}\n")

parsed_doc = parser.parse_pdf(sample_pdf)

# Display summary
print("=" * 60)
print("PARSING RESULTS")
print("=" * 60)
print(f"Filename:        {parsed_doc.filename}")
print(f"Total pages:     {parsed_doc.total_pages}")
print(f"Total chunks:    {parsed_doc.total_chunks}")
print(f"Chunks per page: {parsed_doc.chunks_per_page:.2f}")
print(f"File size:       {parsed_doc.file_size_bytes / 1024:.2f} KB")
print(f"Processing time: {parsed_doc.processing_time_seconds}s")
print("=" * 60)

Parsing: Resumen_8_reunion_grupo_motor_noviembre2025.pdf

PARSING RESULTS
Filename:        Resumen_8_reunion_grupo_motor_noviembre2025.pdf
Total pages:     8
Total chunks:    51
Chunks per page: 6.38
File size:       465.13 KB
Processing time: 0.81s


### Inspect Chunks

Each chunk is a **DocumentChunk** Pydantic model with:
- `text`: The chunk content
- `document`: Source PDF filename
- `page`: Page number (starts at 1)
- `chunk_index`: Sequential chunk number (starts at 0)

In [6]:
# Show first 3 chunks
print("First 3 chunks:\n")

for i in range(min(3, len(parsed_doc.chunks))):
    chunk = parsed_doc.chunks[i]
    print(f"{'='*60}")
    print(f"Chunk #{chunk.chunk_index}")
    print(f"{'='*60}")
    print(f"Document: {chunk.document}")
    print(f"Page:     {chunk.page}")
    print(f"Length:   {len(chunk.text)} characters")
    print(f"\nText preview (first 200 chars):")
    print(f"{chunk.text[:200]}...")
    print()

First 3 chunks:

Chunk #0
Document: Resumen_8_reunion_grupo_motor_noviembre2025.pdf
Page:     1
Length:   143 characters

Text preview (first 200 chars):
RESUMEN DE REUNIÓN DE 6 DE NOVIEMBRE DE 2025 DEL GRUPO MOTOR PARA EL
SEGUIMIENTO DEL CUARTO PLAN DE GOBIERNO ABIERTO DEL AYUNTAMIENTO DE MADRID...

Chunk #1
Document: Resumen_8_reunion_grupo_motor_noviembre2025.pdf
Page:     1
Length:   486 characters

Text preview (first 200 chars):
El día 6 de noviembre de 2025 tiene lugar una nueva reunión del grupo motor para el seguimiento
del Cuarto Plan de Gobierno Abierto del Ayuntamiento de Madrid.
Asisten a la reunión las personas que se...

Chunk #2
Document: Resumen_8_reunion_grupo_motor_noviembre2025.pdf
Page:     1
Length:   269 characters

Text preview (first 200 chars):
Al inicio de la reunión el grupo traslada el pésame a la representante de CERM I por el fallecimiento
del Presidente de su organización y manifiesta su agradecimiento por su trabajo y su participación...



### Example: Single Chunk as JSON

Pydantic models serialize easily to JSON (useful for APIs and storage).

In [7]:
# Serialize one chunk to JSON
example_chunk = parsed_doc.chunks[0]

print("DocumentChunk as JSON:\n")
print(example_chunk.model_dump_json(indent=2))

DocumentChunk as JSON:

{
  "text": "RESUMEN DE REUNIÓN DE 6 DE NOVIEMBRE DE 2025 DEL GRUPO MOTOR PARA EL\nSEGUIMIENTO DEL CUARTO PLAN DE GOBIERNO ABIERTO DEL AYUNTAMIENTO DE MADRID",
  "document": "Resumen_8_reunion_grupo_motor_noviembre2025.pdf",
  "page": 1,
  "chunk_index": 0
}


### Chunk Statistics

Analyze chunk length distribution to verify chunking strategy.

In [8]:
# Calculate chunk length statistics
chunk_lengths = [len(chunk.text) for chunk in parsed_doc.chunks]

print("Chunk Length Statistics:")
print(f"  Min length:     {min(chunk_lengths)} chars")
print(f"  Max length:     {max(chunk_lengths)} chars")
print(f"  Average length: {sum(chunk_lengths) / len(chunk_lengths):.1f} chars")
print(f"  Target size:    {config.chunk_size} chars")
print(f"\nNote: Chunks may be smaller than target to respect semantic boundaries.")

Chunk Length Statistics:
  Min length:     47 chars
  Max length:     509 chars
  Average length: 381.7 chars
  Target size:    512 chars

Note: Chunks may be smaller than target to respect semantic boundaries.


### Filter Chunks by Page

Useful for debugging or viewing specific page content.

In [9]:
# Get all chunks from page 1
page_1_chunks = parsed_doc.get_chunks_by_page(page=1)

print(f"Chunks from page 1: {len(page_1_chunks)}\n")

for chunk in page_1_chunks:
    print(f"Chunk #{chunk.chunk_index} - {len(chunk.text)} chars")
    print(f"  {chunk.text[:100]}...\n")

Chunks from page 1: 7

Chunk #0 - 143 chars
  RESUMEN DE REUNIÓN DE 6 DE NOVIEMBRE DE 2025 DEL GRUPO MOTOR PARA EL
SEGUIMIENTO DEL CUARTO PLAN DE ...

Chunk #1 - 486 chars
  El día 6 de noviembre de 2025 tiene lugar una nueva reunión del grupo motor para el seguimiento
del ...

Chunk #2 - 269 chars
  Al inicio de la reunión el grupo traslada el pésame a la representante de CERM I por el fallecimient...

Chunk #3 - 457 chars
  Seguimiento del IV Plan de Gobierno Abierto
De acuerdo con la propuesta realizada en la reunión prev...

Chunk #4 - 89 chars
  Ciudad de Madrid; 10 del Consejo Sectorial de Asociaciones y otras
Entidades Ciudadanas ....

Chunk #5 - 499 chars
  - Publicación de la informació n relevante en el portal de transparencia . Ángela Pérez
resume los d...

Chunk #6 - 198 chars
  reuniones de seguimiento , el avance en el cumplimiento de los hitos previstos y las
actuaciones de ...



---
### Parse All PDFs

Now let's process all 9 documents to build our complete knowledge base (we should build it document by document if they don't fit in memory at the same time)

In [10]:
# Parse all PDFs
print("Parsing all PDFs...\n")

all_parsed_docs = []

for i, pdf_path in enumerate(pdf_files, 1):
    print(f"[{i}/{len(pdf_files)}] Parsing {pdf_path.name}...", end=" ")
    
    parsed = parser.parse_pdf(pdf_path)
    all_parsed_docs.append(parsed)
    
    print(f"✓ ({parsed.total_pages} pages, {parsed.total_chunks} chunks, {parsed.processing_time_seconds}s)")

print(f"\n✓ All PDFs parsed successfully!")

Parsing all PDFs...

[1/9] Parsing Resumen_8_reunion_grupo_motor_noviembre2025.pdf... ✓ (8 pages, 51 chunks, 1.175s)
[2/9] Parsing Resumen_9_reunion_marzo2026.pdf... ✓ (6 pages, 41 chunks, 0.431s)
[3/9] Parsing Resumen_grupo-motor130924.pdf... ✓ (10 pages, 56 chunks, 0.828s)
[4/9] Parsing resumen_grupo-motor240520.pdf... ✓ (4 pages, 27 chunks, 0.25s)
[5/9] Parsing resumen_grupo_motor110424.pdf... ✓ (40 pages, 280 chunks, 1.781s)
[6/9] Parsing Resumen_grupo_motor_041024.pdf... ✓ (3 pages, 19 chunks, 0.103s)
[7/9] Parsing Resumen_grupo_motor_14032025.pdf... ✓ (6 pages, 43 chunks, 0.229s)
[8/9] Parsing resumengrupomotor260224.pdf... ✓ (12 pages, 59 chunks, 0.506s)
[9/9] Parsing ResumenReunionGA_20231124.pdf... ✓ (9 pages, 43 chunks, 0.285s)

✓ All PDFs parsed successfully!


### Overall Statistics

In [11]:
# Aggregate statistics
total_pages = sum(doc.total_pages for doc in all_parsed_docs)
total_chunks = sum(doc.total_chunks for doc in all_parsed_docs)
total_processing_time = sum(doc.processing_time_seconds for doc in all_parsed_docs)

print("=" * 60)
print("COMPLETE CORPUS STATISTICS")
print("=" * 60)
print(f"Total documents:      {len(all_parsed_docs)}")
print(f"Total pages:          {total_pages}")
print(f"Total chunks:         {total_chunks}")
print(f"Average chunks/doc:   {total_chunks / len(all_parsed_docs):.1f}")
print(f"Average chunks/page:  {total_chunks / total_pages:.2f}")
print(f"Total processing:     {total_processing_time:.2f}s")
print("=" * 60)

COMPLETE CORPUS STATISTICS
Total documents:      9
Total pages:          98
Total chunks:         619
Average chunks/doc:   68.8
Average chunks/page:  6.32
Total processing:     5.59s


### Extract All Chunk Texts

Prepare texts for embedding generation (next step).

In [12]:
# Collect all chunks from all documents
all_chunks = []
for doc in all_parsed_docs:
    all_chunks.extend(doc.chunks)

# Extract texts
all_texts = [chunk.text for chunk in all_chunks]

print(f"Extracted {len(all_texts)} text chunks ready for embedding.")
print(f"\nExample texts[0][:200]:\n{all_texts[0][:200]}...")

Extracted 619 text chunks ready for embedding.

Example texts[0][:200]:
RESUMEN DE REUNIÓN DE 6 DE NOVIEMBRE DE 2025 DEL GRUPO MOTOR PARA EL
SEGUIMIENTO DEL CUARTO PLAN DE GOBIERNO ABIERTO DEL AYUNTAMIENTO DE MADRID...


---
## 2. Embedding Generation

### Model Overview

We use **sentence-transformers** with the `paraphrase-multilingual-MiniLM-L12-v2` model:
- **Multilingual**: Supports Spanish and 50+ languages
- **Free & Local**: No API keys, works offline
- **Lightweight**: 384 dimensions, ~120MB download
- **Fast**: CPU-friendly, no GPU required
- **Quality**: Good semantic search performance for RAG

### Embedder Configuration

Configure the embedding model with batch processing for efficiency.

In [13]:
# Create embedder with configuration
embedding_config = EmbeddingConfig(
    model_name="paraphrase-multilingual-MiniLM-L12-v2",
    device="cpu",
    batch_size=64,
    normalize_embeddings=True,  # Important for cosine similarity
    show_progress=True          # Show progress bar
)

embedder = Embedder(config=embedding_config)

print("Embedder Configuration:")
print(json.dumps(embedder.get_stats(), indent=2))

Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2...


c:\Development\environments\aida-venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✓ Model loaded (dimension: 384)
Embedder Configuration:
{
  "model_name": "paraphrase-multilingual-MiniLM-L12-v2",
  "dimension": 384,
  "device": "cpu",
  "batch_size": 64,
  "normalized": true
}


### Test Embedder with Sample Texts

Let's verify the embedder works correctly with Spanish text.

In [15]:
# Test with sample Spanish texts
test_texts = [
    "El presupuesto aprobado fue de 5 millones de euros",
    "La reunión se celebró el 6 de noviembre",
    "El grupo motor analizó el progreso del plan"
]

print("Embedding sample texts...\n")

test_embeddings = embedder.embed_documents(test_texts)

print(f"Generated embeddings:")
print(f"  Number of texts:     {len(test_texts)}")
print(f"  Embeddings shape:    {test_embeddings.shape}")
print(f"  Embedding dimension: {embedder.dimension}")
print(f"  Data type:           {test_embeddings.dtype}")

Embedding sample texts...



Embedding documents: 100%|██████████| 1/1 [00:00<00:00, 10.78batch/s]

Generated embeddings:
  Number of texts:     3
  Embeddings shape:    (3, 384)
  Embedding dimension: 384
  Data type:           float32


### Semantic Similarity Test

Verify the model captures semantic relationships in Spanish.

In [16]:
# Test semantic similarity with related vs unrelated texts
text1 = "El presupuesto aprobado fue de 5 millones"
text2 = "Se aprobó un presupuesto de 5M de euros"  # Related: same topic
text3 = "El clima está soleado hoy"                # Unrelated: different topic

emb1 = embedder.embed_query(text1)
emb2 = embedder.embed_query(text2)
emb3 = embedder.embed_query(text3)

# Cosine similarity (normalized vectors = dot product)
sim_related = np.dot(emb1, emb2)
sim_unrelated = np.dot(emb1, emb3)

print("Semantic Similarity Test:")
print("=" * 60)
print(f"Text 1: \"{text1}\"")
print(f"Text 2: \"{text2}\"")
print(f"Text 3: \"{text3}\"")
print("=" * 60)
print(f"Similarity (Text 1 ↔ Text 2): {sim_related:.3f}  ← Should be HIGH")
print(f"Similarity (Text 1 ↔ Text 3): {sim_unrelated:.3f}  ← Should be LOW")
print("=" * 60)

if sim_related > 0.7 and sim_unrelated < 0.5:
    print("✓ Semantic similarity test PASSED")
else:
    print("⚠ Unexpected similarity scores")

Semantic Similarity Test:
Text 1: "El presupuesto aprobado fue de 5 millones"
Text 2: "Se aprobó un presupuesto de 5M de euros"
Text 3: "El clima está soleado hoy"
Similarity (Text 1 ↔ Text 2): 0.875  ← Should be HIGH
Similarity (Text 1 ↔ Text 3): 0.032  ← Should be LOW
✓ Semantic similarity test PASSED


### Embed Single Chunk (Example)

Show what a single chunk embedding looks like.

In [17]:
# Embed one chunk as example
example_chunk_text = all_chunks[0].text
example_embedding = embedder.embed_query(example_chunk_text)

print("Example Chunk:")
print("=" * 60)
print(f"Text: {example_chunk_text[:150]}...")
print(f"\nDocument: {all_chunks[0].document}")
print(f"Page:     {all_chunks[0].page}")
print("=" * 60)
print(f"\nEmbedding:")
print(f"  Shape:        {example_embedding.shape}")
print(f"  Dimension:    {len(example_embedding)}")
print(f"  Type:         {example_embedding.dtype}")
print(f"  Normalized:   {np.isclose(np.linalg.norm(example_embedding), 1.0)}")
print(f"\nFirst 10 values: {example_embedding[:10]}")

Example Chunk:
Text: RESUMEN DE REUNIÓN DE 6 DE NOVIEMBRE DE 2025 DEL GRUPO MOTOR PARA EL
SEGUIMIENTO DEL CUARTO PLAN DE GOBIERNO ABIERTO DEL AYUNTAMIENTO DE MADRID...

Document: Resumen_8_reunion_grupo_motor_noviembre2025.pdf
Page:     1

Embedding:
  Shape:        (384,)
  Dimension:    384
  Type:         float32
  Normalized:   True

First 10 values: [-0.052811    0.06470341  0.05471934 -0.01621054  0.04527275  0.02462939
 -0.0426808   0.03046308 -0.03520447 -0.03318403]


---
### Embed All Chunks

Now generate embeddings for all 619 chunks from our corpus.

In [ ]:
# Embed all chunks with progress bar
print(f"Embedding {len(all_texts)} chunks...\n")

all_embeddings = embedder.embed_documents(all_texts)

print(f"\n{'='*60}")
print("EMBEDDING RESULTS")
print(f"{'='*60}")
print(f"Total chunks embedded:  {len(all_embeddings)}")
print(f"Embedding shape:        {all_embeddings.shape}")
print(f"Embedding dimension:    {embedder.dimension}")
print(f"Memory size:            {all_embeddings.nbytes / (1024*1024):.2f} MB")
print(f"{'='*60}")

### Verify Embeddings

Check that all embeddings are properly normalized and have correct dimensions.

In [ ]:
# Verify embeddings quality
norms = np.linalg.norm(all_embeddings, axis=1)

print("Embedding Verification:")
print("=" * 60)
print(f"Min norm:     {norms.min():.6f}")
print(f"Max norm:     {norms.max():.6f}")
print(f"Mean norm:    {norms.mean():.6f}")
print(f"Std norm:     {norms.std():.6f}")
print("=" * 60)

# Check normalization (should be close to 1.0)
if np.allclose(norms, 1.0, atol=1e-5):
    print("✓ All embeddings are properly L2-normalized")
else:
    print("⚠ Warning: Some embeddings may not be normalized")

# Check for NaN or Inf values
if not np.any(np.isnan(all_embeddings)) and not np.any(np.isinf(all_embeddings)):
    print("✓ No NaN or Inf values detected")
else:
    print("⚠ Warning: NaN or Inf values found")

### Embedding Statistics by Document

Analyze embeddings per document to ensure even distribution.

In [ ]:
# Show embeddings per document
print("Embeddings per Document:")
print("=" * 60)

for doc in all_parsed_docs:
    doc_chunks = doc.total_chunks
    doc_size_mb = doc_chunks * embedder.dimension * 4 / (1024 * 1024)  # 4 bytes per float32
    print(f"{doc.filename:<50} {doc_chunks:>3} chunks ({doc_size_mb:.2f} MB)")

print("=" * 60)

### Sample Similarity Matrix

Show similarity between first few chunks to visualize semantic relationships.

In [ ]:
# Compute similarity matrix for first 5 chunks
n_sample = 5
sample_embeddings = all_embeddings[:n_sample]
similarity_matrix = np.dot(sample_embeddings, sample_embeddings.T)

print(f"Similarity Matrix (first {n_sample} chunks):")
print("=" * 60)
print("     ", end="")
for i in range(n_sample):
    print(f"  C{i}  ", end="")
print()

for i in range(n_sample):
    print(f"C{i}  ", end="")
    for j in range(n_sample):
        print(f" {similarity_matrix[i,j]:.3f}", end="")
    print()

print("=" * 60)
print("Note: Diagonal should be 1.0 (chunk compared to itself)")
print("      Higher values indicate more similar chunks")

In [ ]:
# Ready for vector storage
print("Ready for next phase: Vector Storage (FAISS)")
print(f"Data available:")
print(f"  - all_chunks: {len(all_chunks)} DocumentChunk objects")
print(f"  - all_embeddings: {all_embeddings.shape} NumPy array")
print(f"  - embedder.dimension: {embedder.dimension}")